# 📐 VideoRAG — ITC LoRA Fine-tuning (04_lora_finetune)

## 목적

InternVideo2 BERT text encoder의 CLS collapse 문제를 LoRA fine-tuning으로 완화한다.

**진단 근거 (10차 이슈 보고서)**
- layer 0: cosine = 1.0000 (CLS token embedding, 모든 문장 동일)
- layer 6: cosine = 0.9254 (최저점, 그나마 분화)
- layer 7~18: cosine 다시 0.99대로 수렴 ← **문제 구간**
- layer 19: cosine = 0.9330 (최종 출력)

**전략**
- layer 7~18의 Q, K, V projection에 LoRA(rank=8) 삽입
- layer 6에서 분화된 신호를 layer 7 이후에서 죽이지 않도록 학습
- MSR-VTT 1k-A ITC contrastive loss로 파인튜닝
- Video encoder, layer 0~6: frozen

**평가**
- 파인튜닝 전/후 CLS cosine 레이어별 측정
- R@1 변화 확인

## Step 0: 환경 부트스트랩

In [ ]:
# ── Step 0: 환경 부트스트랩 (런타임 초기화 시 1회) ──

import os, shutil
os.chdir('/content')
from google.colab import userdata

# (1) Google Drive 마운트
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print('✓ Drive 마운트 완료')
else:
    print('✓ Drive 이미 마운트됨')

# (2) 프로젝트 코드 복사
DRIVE_PROJECT = '/content/drive/MyDrive/videorag_prototype'
LOCAL_PROJECT = '/content/videorag_prototype'

if not os.path.exists(LOCAL_PROJECT):
    if os.path.exists(DRIVE_PROJECT):
        shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
        print('✓ Drive → 로컬 복사 완료')
    else:
        print(f'⚠ {DRIVE_PROJECT} 없음 → git clone 필요')
else:
    print(f'✓ 프로젝트 이미 존재: {LOCAL_PROJECT}')

# (2-1) GitHub에서 최신 src 동기화
os.system(f'rm -rf /tmp/VideoRAG-Prototype')
os.system('git clone https://github.com/LimPark996/VideoRAG-Public.git /tmp/VideoRAG-Prototype')
os.chdir('/content')
os.system(f'rm -rf {LOCAL_PROJECT}/src')
os.system(f'cp -r /tmp/VideoRAG-Prototype/src {LOCAL_PROJECT}/src')
print('✓ src 동기화 완료')

# (3) 의존성 설치
!pip install -q open_clip_torch 2>/dev/null
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118 2>/dev/null
!pip install -q transformers timm einops rank_bm25 faiss-cpu \
    moviepy opencv-python cryptography scikit-learn scipy \
    tqdm matplotlib pandas numpy easydict langdetect requests 2>/dev/null
!pip install -q peft 2>/dev/null  # LoRA용
print('✓ 의존성 설치 완료 (peft 포함)')

# (4) sys.path 등록
import sys
sys.path.insert(0, LOCAL_PROJECT)
print(f'✓ sys.path에 {LOCAL_PROJECT} 추가')

# (5) HF_TOKEN
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('✓ HF_TOKEN 설정 완료')

## Step 0.5: InternVideo2 설치

In [ ]:
# ── Step 0.5: InternVideo2 설치 (런타임 초기화 시 1회) ──
import shutil, os, sys, importlib

if os.path.exists('/content/InternVideo'):
    shutil.rmtree('/content/InternVideo')
    print('✓ 기존 /content/InternVideo 삭제')

!git clone --no-checkout --depth=1 https://github.com/OpenGVLab/InternVideo.git /content/InternVideo
%cd /content/InternVideo
!git sparse-checkout init --cone
!git sparse-checkout set InternVideo2/multi_modality
!git checkout main

INTERNVIDEO_PATH = '/content/InternVideo/InternVideo2/multi_modality'
if INTERNVIDEO_PATH not in sys.path:
    sys.path.insert(0, INTERNVIDEO_PATH)
importlib.invalidate_caches()
print(f'✓ InternVideo2 설치 완료: {INTERNVIDEO_PATH}')

## Step 1: InternVideo2 모델 로드

In [ ]:
# ── Step 1: 모델 로드 ──
import sys, os, torch

LOCAL_PROJECT = '/content/videorag_prototype'
INTERNVIDEO_PATH = '/content/InternVideo/InternVideo2/multi_modality'
sys.path.insert(0, LOCAL_PROJECT)
sys.path.insert(0, INTERNVIDEO_PATH)

from src.pipeline import VideoRAGPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

pipeline = VideoRAGPipeline(device=device)
iv_model = pipeline.iv_model
print('✓ InternVideo2 로드 완료')

# BERT text encoder 확인
bert = iv_model.get_text_encoder()
print(f'num_hidden_layers: {bert.config.num_hidden_layers}')
print(f'fusion_layer: {bert.config.fusion_layer}')
print(f'→ mode=text에서 실행되는 레이어: 0 ~ {bert.config.fusion_layer - 1}')

## Step 2: 베이스라인 — 레이어별 CLS cosine 측정

파인튜닝 전 상태를 기록한다.

In [ ]:
# ── Step 2: 베이스라인 CLS cosine 측정 ──
import torch, torch.nn.functional as F
import urllib.request, json, numpy as np

# 1k-A annotation 로드
ANN_DIR  = os.path.join(LOCAL_PROJECT, 'data', 'msrvtt', 'annotations')
ANN_PATH = os.path.join(ANN_DIR, 'msrvtt_test_1k.json')
os.makedirs(ANN_DIR, exist_ok=True)
if not os.path.exists(ANN_PATH):
    url = 'https://huggingface.co/datasets/YuanhaoGe/MSRVTT-TestData/resolve/main/MSRVTT_JSFUSION_test.json'
    urllib.request.urlretrieve(url, ANN_PATH)
    print('✓ annotation 다운로드 완료')

with open(ANN_PATH) as f:
    ann = json.load(f)

# 캡션 200개 샘플링
captions = [item['caption'] for item in ann['annotations']][:200]
print(f'샘플 캡션 {len(captions)}개 준비')

def measure_layerwise_cls_cosine(bert_model, tokenizer, captions, max_txt_l, device, label=''):
    """
    BERT 레이어별 CLS cosine 평균을 측정한다.
    반환: dict {layer_idx: cosine_mean}
    """
    tok = tokenizer(
        captions,
        padding='max_length',
        truncation=True,
        max_length=max_txt_l,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        output = bert_model(
            input_ids=tok.input_ids,
            attention_mask=tok.attention_mask,
            output_hidden_states=True,
            return_dict=True,
            mode='text'
        )

    hidden_states = output.hidden_states  # 튜플
    n = len(captions)
    results = {}

    for i, hs in enumerate(hidden_states):
        cls = hs[:, 0].float()
        cls_norm = F.normalize(cls, dim=-1)
        cos = (cls_norm @ cls_norm.T)
        cos.fill_diagonal_(0)
        mean_cos = (cos.sum() / (n * (n - 1))).item()
        results[i] = mean_cos

    if label:
        print(f'\n=== {label} ===')
        for layer_i, val in results.items():
            marker = ' ← 최저점' if val == min(results.values()) else ''
            print(f'  layer {layer_i:2d}: {val:.4f}{marker}')

    return results

# 베이스라인 측정
baseline_cosine = measure_layerwise_cls_cosine(
    bert_model=bert,
    tokenizer=iv_model.tokenizer,
    captions=captions,
    max_txt_l=iv_model.config.max_txt_l,
    device=device,
    label='베이스라인 (파인튜닝 전)'
)

## Step 3: LoRA 설정

**적용 위치**: layer 7~18의 Q, K, V projection

**이유**:
- layer 6에서 cosine = 0.9254 (최저점, 분화 최대)
- layer 7부터 다시 0.99대로 수렴 → 분화를 죽이는 구간
- Q, K, V에 LoRA를 붙이면 attention 패턴이 바뀌어
  CLS가 문장 내용을 더 잘 흡수하는 방향으로 학습 가능

**frozen**:
- Video encoder
- BERT layer 0~6
- BERT layer 7~18의 원본 Wq, Wk, Wv (LoRA의 delta만 학습)

In [ ]:
# ── Step 3: LoRA 설정 ──
from peft import LoraConfig, get_peft_model, TaskType

# BERT Q, K, V projection 모듈 이름 확인
# InternVideo2 BERT는 HuggingFace BertModel 계열
# layer명: bert.encoder.layer.{i}.attention.self.query / .key / .value
print('BERT 모듈 구조 샘플 확인:')
for name, module in bert.named_modules():
    if 'attention.self' in name and any(x in name for x in ['query', 'key', 'value']):
        if 'encoder.layer.7.' in name or 'encoder.layer.8.' in name:
            print(f'  {name}: {module.__class__.__name__} | shape: {list(module.weight.shape)}')

# target_modules: layer 7~18의 query, key, value
# peft의 target_modules는 모듈 이름의 마지막 부분(suffix)을 매칭
# → 전체 레이어에 적용되므로, layer 0~6은 수동으로 freeze

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_LAYERS = list(range(7, 19))  # layer 7~18

print(f'\nLoRA 설정:')
print(f'  rank: {LORA_RANK}')
print(f'  alpha: {LORA_ALPHA}')
print(f'  dropout: {LORA_DROPOUT}')
print(f'  target layers: {TARGET_LAYERS}')

## Step 4: 수동 LoRA 삽입 (layer 7~18 Q, K, V)

In [ ]:
# ── Step 4: 수동 LoRA 삽입 ──
#
# peft의 get_peft_model은 target_modules를 suffix로 매칭해서
# 전체 레이어에 적용된다. layer 7~18만 선택적으로 적용하려면
# 수동으로 LoRA Linear를 교체하고 나머지는 freeze하는 방법을 쓴다.

import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    원본 Linear weight는 frozen.
    delta = A @ B 만 학습.

    forward:
        output = x @ W.T + x @ A.T @ B.T * (alpha / rank)
    """
    def __init__(self, original_linear, rank=8, alpha=16):
        super().__init__()
        self.original = original_linear
        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False

        in_features  = original_linear.in_features
        out_features = original_linear.out_features
        self.rank  = rank
        self.scale = alpha / rank

        # A: [in_features, rank], B: [rank, out_features]
        self.lora_A = nn.Parameter(torch.empty(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

        # A는 kaiming uniform, B는 0으로 초기화 → 초기 delta=0
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        base_out = self.original(x)
        # delta = x @ A @ B * scale
        lora_out = (x @ self.lora_A @ self.lora_B) * self.scale
        return base_out + lora_out


def insert_lora_to_bert(bert_model, target_layers, rank=8, alpha=16):
    """
    bert_model의 encoder.layer.{i}.attention.self.{query,key,value}를
    target_layers에 해당하는 것만 LoRALinear로 교체.
    나머지는 전부 freeze.
    """
    # 먼저 전체 freeze
    for param in bert_model.parameters():
        param.requires_grad = False

    replaced = []
    for layer_idx in target_layers:
        attn_self = bert_model.encoder.layer[layer_idx].attention.self
        for proj_name in ['query', 'key', 'value']:
            original = getattr(attn_self, proj_name)
            lora_layer = LoRALinear(original, rank=rank, alpha=alpha)
            setattr(attn_self, proj_name, lora_layer)
            replaced.append(f'layer.{layer_idx}.attention.self.{proj_name}')

    return replaced


# LoRA 삽입 실행
replaced = insert_lora_to_bert(bert, TARGET_LAYERS, rank=LORA_RANK, alpha=LORA_ALPHA)
print(f'LoRA 삽입 완료: {len(replaced)}개 projection')
for r in replaced:
    print(f'  ✓ {r}')

# 학습 파라미터 수 확인
trainable = sum(p.numel() for p in bert.parameters() if p.requires_grad)
total     = sum(p.numel() for p in bert.parameters())
print(f'\n학습 파라미터: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

## Step 5: 학습 데이터셋 준비 (msrvtt_metadata_full.csv, 7,010쌍)

In [ ]:
# ── Step 5: 학습 데이터셋 준비 ──
#
# msrvtt_metadata_full.csv: video_id, caption, video_path
# 7,010행 (train+val 전체)
# 1k-A test split은 평가용이므로 학습에 쓰지 않는다.

import pandas as pd
from torch.utils.data import Dataset, DataLoader

METADATA_CSV = '/content/drive/MyDrive/videorag_prototype/data/msrvtt/videos/msrvtt_captions/msrvtt_metadata_full.csv'

df = pd.read_csv(METADATA_CSV)
print(f'CSV 로드 완료: {len(df)}행')
print(f'컬럼: {list(df.columns)}')
print(df.head(3))

# (video_id, caption, video_path) 튜플 리스트
train_pairs = list(zip(df['video_id'].tolist(), df['caption'].tolist(), df['video_path'].tolist()))
print(f'\n학습 쌍: {len(train_pairs)}개')


class MSRVTTTrainDataset(Dataset):
    """
    (video_id, caption, video_path) 반환.
    video feature는 Step 6에서 사전 계산된 캐시에서 로드.
    """
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        video_id, caption, video_path = self.pairs[idx]
        return video_id, caption, video_path


dataset = MSRVTTTrainDataset(train_pairs)
loader  = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
print(f'DataLoader: {len(loader)} 배치 (batch_size=32)')

## Step 6: Video Feature 캐시 빌드

01_indexing에서 이미 생성된 FAISS 인덱스를 Drive에서 직접 읽는다.  
**실행 전 확인**: `DRIVE_INDEX` 경로가 7,010개짜리 인덱스를 가리키는지 확인할 것.  
`index_demo` (7,010개) vs `index_eval` (1,000개)

In [ ]:
# ── Step 6: Video Feature 캐시 빌드 ──
#
# ⚠️  실행 전 확인:
#   DRIVE_INDEX 경로를 Drive에서 직접 확인하세요.
#   index_demo → 7,010개 (학습용)
#   index_eval → 1,000개 (평가용, 학습에 쓰면 안 됨)

import pickle, torch, faiss, numpy as np

# ── 경로 설정 (여기만 수정) ──
DRIVE_INDEX = '/content/drive/MyDrive/videorag_prototype/index_demo'

# ntotal 먼저 확인
faiss_path = os.path.join(DRIVE_INDEX, 'faiss_ivfflat.index')
index = faiss.read_index(faiss_path)
print(f'FAISS 벡터 수: {index.ntotal}')
if index.ntotal < 5000:
    print('⚠️  1,000개짜리 eval 인덱스로 보임 → DRIVE_INDEX를 index_demo로 변경하세요')
    raise ValueError('학습용 인덱스가 아님')
else:
    print('✓ 7,010개 확인 → 학습에 사용 가능')

# clip_metadata 로드
meta_path = os.path.join(DRIVE_INDEX, 'clip_metadata.pkl')
with open(meta_path, 'rb') as f:
    clip_meta = pickle.load(f)

clip_ids = list(clip_meta.keys())
print(f'clip_metadata: {len(clip_ids)}개')
print(f'샘플 clip_id: {clip_ids[0]}')
print(f'샘플 메타: {clip_meta[clip_ids[0]]}')

# FAISS에서 전체 벡터 추출 (Drive에서 직접)
vecs = index.reconstruct_n(0, index.ntotal)  # [N, 512] numpy
print(f'벡터 추출 완료: {vecs.shape}')

# video_id → 벡터 딕셔너리 구성
# 샘플 메타 보고 video_id 키 이름 확인 후 필요시 수정
video_feat_cache = {}
for i, clip_id in enumerate(clip_ids):
    meta = clip_meta[clip_id]
    # ⚠️ 아래 키 이름은 샘플 메타 출력 보고 맞게 수정
    video_id = meta.get('video_id', meta.get('clip_id', clip_id))
    video_feat_cache[video_id] = torch.tensor(vecs[i], dtype=torch.float32)

print(f'\nvideo_feat_cache 구성 완료: {len(video_feat_cache)}개')
sample_vid = list(video_feat_cache.keys())[0]
print(f'샘플: {sample_vid} → shape {video_feat_cache[sample_vid].shape}')

# train_pairs와 매칭 확인
missing = [vid for vid, _, _ in train_pairs if vid not in video_feat_cache]
print(f'\ntrain_pairs 중 캐시 없는 video_id: {len(missing)}개')
if missing[:3]:
    print(f'  예시: {missing[:3]} ← 키 이름 불일치일 수 있음')

## Step 7: ITC Contrastive Loss 정의

In [ ]:
# ── Step 7: ITC Contrastive Loss 정의 ──
#
# InfoNCE (NT-Xent) contrastive loss
# 같은 쌍(video_i, text_i) → similarity 높게
# 다른 쌍(video_i, text_j) → similarity 낮게
#
# 배치 내 negative를 활용하는 표준 구현

import torch
import torch.nn.functional as F

def itc_contrastive_loss(video_emb, text_emb, temperature=0.07):
    """
    video_emb: [B, D] L2 normalized
    text_emb:  [B, D] L2 normalized
    temperature: softmax temperature (논문 기본값 0.07)

    반환: scalar loss
    """
    # [B, B] similarity matrix
    sim_matrix = (video_emb @ text_emb.T) / temperature

    # 정답은 diagonal (video_i ↔ text_i)
    labels = torch.arange(len(video_emb), device=video_emb.device)

    # video → text 방향
    loss_v2t = F.cross_entropy(sim_matrix, labels)
    # text → video 방향
    loss_t2v = F.cross_entropy(sim_matrix.T, labels)

    return (loss_v2t + loss_t2v) / 2


# 동작 확인
dummy_v = F.normalize(torch.randn(4, 512), dim=-1)
dummy_t = F.normalize(torch.randn(4, 512), dim=-1)
test_loss = itc_contrastive_loss(dummy_v, dummy_t)
print(f'Loss 함수 동작 확인: {test_loss.item():.4f} (random 입력 기준 ~log(4)={torch.log(torch.tensor(4.)):.4f}와 유사해야 함)')

## Step 8: 텍스트 인코딩 함수 (LoRA 포함)

In [ ]:
# ── Step 8: 텍스트 인코딩 함수 ──
#
# LoRA가 삽입된 BERT를 통해 텍스트를 인코딩한다.
# mean pooling 방식 사용 (CLS collapse 우회 + LoRA 학습 병행)

def encode_text_with_lora(bert_model, text_proj, tokenizer, captions, max_txt_l, device):
    """
    LoRA가 삽입된 BERT로 텍스트 인코딩.
    mean pooling → text_proj → L2 normalize

    반환: [B, 512] L2 normalized
    """
    tok = tokenizer(
        captions,
        padding='max_length',
        truncation=True,
        max_length=max_txt_l,
        return_tensors='pt'
    ).to(device)

    output = bert_model(
        input_ids=tok.input_ids,
        attention_mask=tok.attention_mask,
        output_hidden_states=False,
        return_dict=True,
        mode='text'
    )

    # hidden_states: [B, 40, 1024]
    hidden = output.last_hidden_state

    # mean pooling (PAD 제외)
    mask = tok.attention_mask.unsqueeze(-1).float()
    mean_pooled = (hidden * mask).sum(1) / mask.sum(1)  # [B, 1024]

    # text_proj
    projected = text_proj(mean_pooled)  # [B, 512]
    return F.normalize(projected, dim=-1)


# text_proj 모듈 추출
text_proj = iv_model.text_proj  # Linear(1024, 512)
text_proj.requires_grad_(False)  # proj는 frozen
print(f'text_proj: {text_proj}')

# 동작 확인
with torch.no_grad():
    test_emb = encode_text_with_lora(bert, text_proj, iv_model.tokenizer, captions[:4], iv_model.config.max_txt_l, device)
print(f'텍스트 인코딩 동작 확인: shape={test_emb.shape}, norm={test_emb.norm(dim=-1).mean():.4f}')

## Step 9: LoRA 학습 루프

**예상 소요시간**: A100 기준 약 20~30분 (3 epoch, 32개 배치)

In [ ]:
# ── Step 9: LoRA 학습 루프 ──
import torch.optim as optim
from tqdm import tqdm

# 하이퍼파라미터
EPOCHS    = 3
LR        = 1e-4
LOG_EVERY = 10  # 배치

# optimizer: LoRA 파라미터만
lora_params = [p for p in bert.parameters() if p.requires_grad]
optimizer = optim.AdamW(lora_params, lr=LR)
print(f'학습 파라미터 수: {sum(p.numel() for p in lora_params):,}')

bert.train()
loss_history = []

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    valid_batches = 0

    for batch_idx, (captions_batch, video_ids_batch) in enumerate(tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}')):

        # video feature 캐시에서 로드
        video_embs = []
        valid_caps = []
        for cap, vid in zip(captions_batch, video_ids_batch):
            if vid in video_feat_cache:
                video_embs.append(video_feat_cache[vid].to(device))
                valid_caps.append(cap)

        if len(video_embs) < 4:  # 너무 적으면 skip
            continue

        video_emb = torch.stack(video_embs)  # [B, 512]
        video_emb = F.normalize(video_emb, dim=-1)

        # 텍스트 인코딩 (LoRA 포함)
        text_emb = encode_text_with_lora(
            bert, text_proj, iv_model.tokenizer,
            valid_caps, iv_model.config.max_txt_l, device
        )  # [B, 512]

        # ITC loss
        loss = itc_contrastive_loss(video_emb, text_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        valid_batches += 1

        if (batch_idx + 1) % LOG_EVERY == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1} | Loss: {loss.item():.4f}')

    avg_loss = epoch_loss / max(valid_batches, 1)
    loss_history.append(avg_loss)
    print(f'\nEpoch {epoch+1} 완료 | avg loss: {avg_loss:.4f}\n')

print('✓ 학습 완료')

## Step 10: 파인튜닝 후 레이어별 CLS cosine 측정

In [ ]:
# ── Step 10: 파인튜닝 후 CLS cosine 측정 ──
bert.eval()

finetuned_cosine = measure_layerwise_cls_cosine(
    bert_model=bert,
    tokenizer=iv_model.tokenizer,
    captions=captions,
    max_txt_l=iv_model.config.max_txt_l,
    device=device,
    label='파인튜닝 후'
)

# 비교 테이블
print('\n=== 레이어별 CLS cosine 비교 ===')
print(f'{"layer":>8} {"before":>10} {"after":>10} {"delta":>10}')
print('-' * 42)
for i in baseline_cosine:
    before = baseline_cosine[i]
    after  = finetuned_cosine[i]
    delta  = after - before
    marker = ' ← ↓ 개선' if delta < -0.005 else ''
    print(f'{i:>8} {before:>10.4f} {after:>10.4f} {delta:>+10.4f}{marker}')

## Step 11: Loss 곡선 시각화

In [ ]:
# ── Step 11: Loss 시각화 ──
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(loss_history)+1), loss_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('ITC Loss')
plt.title('LoRA Fine-tuning Loss (layer 7~18 Q,K,V)')
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/lora_loss.png', dpi=150)
plt.show()
print('✓ loss 곡선 저장: /content/lora_loss.png')

# before/after cosine 비교 시각화
layers = sorted(baseline_cosine.keys())
before_vals = [baseline_cosine[l] for l in layers]
after_vals  = [finetuned_cosine[l] for l in layers]

plt.figure(figsize=(12, 5))
plt.plot(layers, before_vals, label='Before LoRA', marker='o', alpha=0.7)
plt.plot(layers, after_vals,  label='After LoRA',  marker='s', alpha=0.7)
plt.axvspan(7, 18, alpha=0.1, color='orange', label='LoRA 적용 구간 (layer 7~18)')
plt.xlabel('Layer')
plt.ylabel('CLS cosine (평균)')
plt.title('레이어별 CLS cosine: LoRA 전후 비교')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/cls_cosine_comparison.png', dpi=150)
plt.show()
print('✓ cosine 비교 그래프 저장')

## Step 12: LoRA 가중치 저장

In [ ]:
# ── Step 12: LoRA 가중치 저장 ──
import torch

LORA_SAVE_PATH = '/content/lora_bert_layer7to18_qkv.pt'
DRIVE_LORA     = '/content/drive/MyDrive/videorag_prototype/checkpoints/lora_bert_layer7to18_qkv.pt'

# LoRA 파라미터만 추출해서 저장
lora_state = {
    name: param
    for name, param in bert.named_parameters()
    if param.requires_grad
}
torch.save({
    'lora_state': lora_state,
    'config': {
        'target_layers': TARGET_LAYERS,
        'rank': LORA_RANK,
        'alpha': LORA_ALPHA,
        'epochs': EPOCHS,
        'lr': LR,
        'loss_history': loss_history,
    },
    'baseline_cosine': baseline_cosine,
    'finetuned_cosine': finetuned_cosine,
}, LORA_SAVE_PATH)
print(f'✓ LoRA 저장: {LORA_SAVE_PATH}')

# Drive 백업
os.makedirs(os.path.dirname(DRIVE_LORA), exist_ok=True)
shutil.copy2(LORA_SAVE_PATH, DRIVE_LORA)
print(f'✓ Drive 백업: {DRIVE_LORA}')

print(f'\n저장된 파라미터 수: {len(lora_state)}개 텐서')

## Step 13: (선택) R@1 간이 평가

1k-A 전체에 대한 R@1을 측정한다.  
**주의**: 전체 ITM 파이프라인 없이 ITC mean pooling만으로 평가하는 간이 버전.  
정확한 R@1은 03_evaluation.ipynb에서 측정한다.

In [ ]:
# ── Step 13: R@1 간이 평가 (ITC mean pooling only) ──
bert.eval()

if video_feat_cache and len(video_feat_cache) >= 100:
    # 평가 대상: video_feat_cache에 있는 쌍만
    valid_pairs = [(vid, cap) for vid, cap in eval_pairs if vid in video_feat_cache]
    print(f'평가 대상: {len(valid_pairs)}쌍')

    # 텍스트 전체 인코딩
    all_caps  = [cap for _, cap in valid_pairs]
    all_vids  = [vid for vid, _ in valid_pairs]

    BATCH = 64
    all_text_embs = []
    with torch.no_grad():
        for i in range(0, len(all_caps), BATCH):
            emb = encode_text_with_lora(
                bert, text_proj, iv_model.tokenizer,
                all_caps[i:i+BATCH], iv_model.config.max_txt_l, device
            )
            all_text_embs.append(emb.cpu())
    text_embs = torch.cat(all_text_embs, dim=0)  # [N, 512]

    # 영상 임베딩
    video_embs = torch.stack([video_feat_cache[v] for v in all_vids])  # [N, 512]
    video_embs = F.normalize(video_embs, dim=-1)

    # similarity matrix [N, N]
    sim = text_embs @ video_embs.T

    # R@1 계산
    correct = 0
    for i in range(len(valid_pairs)):
        top1 = sim[i].argmax().item()
        if top1 == i:
            correct += 1

    r1 = correct / len(valid_pairs) * 100
    print(f'\n=== ITC only R@1 (LoRA after) ===')
    print(f'R@1: {r1:.1f}%')
    print(f'(참고: mean pooling ITC only 베이스라인 ~39.1%)')
    print(f'(참고: 논문 ITM 포함 51.9%)')
else:
    print('⚠ video_feat_cache 없음 → R@1 평가 스킵')
    print('  Step 6 먼저 실행하세요')

## 결과 해석 가이드

| 지표 | 기대 방향 | 설명 |
|------|-----------|------|
| layer 7~18 cosine | ↓ 감소 | 분화 개선 |
| layer 6 cosine | 유지 | 원래 분화 구간, 건드리지 않음 |
| ITC R@1 | ↑ 증가 | 실제 검색 성능 개선 |
| Loss | ↓ 감소 | 정상 학습 확인 |

**다음 단계**:
1. 03_evaluation.ipynb에서 ITM 포함 전체 파이프라인으로 R@1 재측정
2. rank=16, 또는 layer 범위 조정 실험
3. MSR-VTT 공식 train split (7,000쌍)으로 재학습